In [1]:
import cv2
import numpy as np
import os

In [2]:
import cv2
import numpy as np

def preprocess_bender_image(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    blurred = cv2.medianBlur(img, 3)


    binary = cv2.adaptiveThreshold(
        blurred,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        blockSize=15, 
        C=12         
    )


    min_component_area = 25 

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)
    filtered = np.zeros_like(binary) 

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]

        if area >= min_component_area:
            filtered[labels == i] = 255 

    return img, filtered

In [3]:
import os
input_folder = "Dataset\\Raw_images"
output_folder = "Dataset\\Preprocessed"



# loop through images 1 to 89
for i in range(1, 90):
    image_name = f"{i}.jpg"   # change extension if needed
    input_path = os.path.join(input_folder, image_name)

    # skip if file doesn't exist
    if not os.path.exists(input_path):
        print(f"Skipping {image_name} (not found)")
        continue

    original, processed = preprocess_bender_image(input_path)

    output_path = os.path.join(output_folder, f"{i}.jpg")

    cv2.imwrite(output_path, processed)

print("Processing complete!")

Processing complete!


In [ ]:
import matplotlib.pyplot as plt

def validate_preprocessing(image_path):
    original, binary = preprocess_bender_image(image_path)
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    axes[0].imshow(original, cmap='gray')
    axes[0].set_title('Original Scan')
    axes[1].imshow(binary, cmap='gray')
    axes[1].set_title('After Processing')
    plt.show()
    

In [ ]:
# import cv2
# import numpy as np
# import os
# import math
# import mahotas



# class TemplateProcessor:
#     def __init__(self, template_dir):
#         self.template_dir = template_dir
#         self.templates = {}
#         self.template_features = {}
    
#     def load_templates(self):
#         """Load all template images"""
#         template_files = sorted(os.listdir(self.template_dir))
        
#         for filename in template_files:
#             if filename.endswith(('.png', '.jpg', '.jpeg')):
#                 template_name = os.path.splitext(filename)[0]
#                 path = os.path.join(self.template_dir, filename)
                

#                 _, binary = preprocess_bender_image(path)
                
#                 contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                
#                 if contours:
#                     main_contour = max(contours, key=cv2.contourArea)
                    
#                     self.templates[template_name] = {
#                         'image': binary,
#                         'contour': main_contour,
#                         'filename': filename
#                     }
                    
#         print(f"Loaded {len(self.templates)} templates")
#         return self.templates
    
#     def extract_template_features(self):
#         for name, template in self.templates.items():
#             contour = template['contour']
            
#             # 1. Hu Moments (rotation, scale, translation invariant)
#             moments = cv2.moments(contour)
#             hu_moments = cv2.HuMoments(moments).flatten()
            
#             # 2. Zernike Moments 
#             img = template['image']

#             h, w = img.shape
#             radius = min(h, w) // 2

#             zernike_moments = mahotas.features.zernike_moments(
#                 img,
#                 radius=radius,
#                 degree=8
#             )            
#             fourier_descriptors = self.calculate_fourier_descriptors(contour, num_descriptors=20)
            
#             area = cv2.contourArea(contour)
#             perimeter = cv2.arcLength(contour, True)
#             hull = cv2.convexHull(contour)
#             hull_area = cv2.contourArea(hull)
#             solidity = area / hull_area if hull_area > 0 else 0
            
#             x, y, w, h = cv2.boundingRect(contour)
#             aspect_ratio = w / h if h > 0 else 0
            
#             self.template_features[name] = {
#                 'hu_moments': hu_moments,
#                 'zernike_moments': zernike_moments,
#                 'fourier_descriptors': fourier_descriptors,
#                 'area': area,
#                 'perimeter': perimeter,
#                 'solidity': solidity,
#                 'aspect_ratio': aspect_ratio,
#                 'bounding_box': (x, y, w, h)
#             }
        
#         return self.template_features
    

    
#     def calculate_fourier_descriptors(self, contour, num_descriptors=20):
#         contour_complex = contour.reshape(-1, 2)
#         complex_coords = contour_complex[:, 0] + 1j * contour_complex[:, 1]
        
#         fourier = np.fft.fft(complex_coords)
        
#         descriptors = fourier[1:num_descriptors+1] / fourier[0]
        
#         descriptors_magnitude = np.abs(descriptors)
        
#         return descriptors_magnitude
    
#     def save_template_features(self, output_path):
#         import pickle
        
#         with open(output_path, 'wb') as f:
#             pickle.dump({
#                 'templates': self.templates,
#                 'features': self.template_features
#             }, f)
        
#         print(f"Template features saved to {output_path}")

In [ ]:
# template_processor = TemplateProcessor('Dataset\\templates')
# templates = template_processor.load_templates()
# features = template_processor.extract_template_features()
# template_processor.save_template_features('Dataset\\templates_Preprocessed\\template_features.pkl')

# import matplotlib.pyplot as plt

# fig, axes = plt.subplots(3, 3, figsize=(12, 10))
# for idx, (name, template) in enumerate(templates.items()):
#     row = idx // 3
#     col = idx % 3
#     axes[row, col].imshow(template['image'], cmap='gray')
#     axes[row, col].set_title(f'Template {name}')
#     axes[row, col].axis('off')

# plt.tight_layout()
# plt.show()

In [2]:

import os
import json
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
import re



def clean_gestalt_string(s):
    if not isinstance(s, str):
        return []
    import re
    s = re.sub(r'[\-\–\—]', ',', s)         
    s = s.replace('/', ',')
    parts = [p.strip() for p in s.split(',')]
    return [p for p in parts if p]
class AnnotationAnalyzer:
    def __init__(self, csv_path, output_dir='visualizations'):
        self.csv_path = csv_path
        self.df = None
        self.feature_distributions = {}
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)

    def load_annotations(self):
        """Load CSV annotations"""
        self.df = pd.read_csv(self.csv_path, encoding='utf-8-sig', on_bad_lines='skip')

        print(f"Dataset shape: {self.df.shape}")
        print(f"Columns: {list(self.df.columns)}")
        print("\nFirst few rows:")
        print(self.df.head())

        missing_summary = self.df.isna().sum().to_dict()
        print("\nMissing values per column:")
        for col, miss in missing_summary.items():
            if miss > 0:
                print(f"  {col}: {miss} ({miss / len(self.df):.1%})")

        return self.df




    def analyze_feature_distributions(self):
        gestalt_col = 'Gestalt Changes'               


        self.df[gestalt_col] = self.df[gestalt_col].apply(clean_gestalt_string)


        mlb_gestalt = MultiLabelBinarizer()
        gestalt_encoded = mlb_gestalt.fit_transform(self.df[gestalt_col])
        gestalt_df = pd.DataFrame(gestalt_encoded, columns=mlb_gestalt.classes_, index=self.df.index)
        gestalt_df = gestalt_df.add_prefix('gestalt_')  

        
        single_cols = {
            'Sequence': 'sequence',
            'Drawing Position': 'drawing_position',
            'Distance Between Drawings': 'distance_between',
            'Contact or Collision': 'contact_collision',
            'Margin Usage': 'margin_usage',
            'Paper Position Change': 'paper_position_change'
        }


        single_df = self.df[['Case Number'] + list(single_cols.keys())].copy()
        single_df.rename(columns=single_cols, inplace=True)

        self.full_labels = pd.concat([single_df, gestalt_df], axis=1)


        print("\nSingle choice distributions:")
        for col in single_cols.values():
            print(f"\n{col}:")
            print(self.full_labels[col].value_counts(dropna=False))

        print("\nGestalt binary indicator frequencies:")
        gestalt_cols = [c for c in self.full_labels.columns if c.startswith('gestalt_')]
        for gcol in gestalt_cols:
            count = self.full_labels[gcol].sum()
            print(f"  {gcol}: {count} ({count/len(self.full_labels)*100:.1f}%)")

        print("\nGeneral observation binary indicator frequencies:")
        obs_cols = [c for c in self.full_labels.columns if c.startswith('obs_')]
        for ocol in obs_cols:
            count = self.full_labels[ocol].sum()
            print(f"  {ocol}: {count} ({count/len(self.full_labels)*100:.1f}%)")

        self.gestalt_column_names = mlb_gestalt.classes_
        return self.full_labels

    def identify_imbalanced_classes(self, threshold=0.10):
        imbalanced_features = {}

        for feature_name, distribution in self.feature_distributions.items():
            total = distribution['total']
            imbalanced_classes = []

            for class_name, count in distribution['counts'].items():
                if class_name == 'Missing':
                    continue

                proportion = count / total
                if proportion < threshold:
                    imbalanced_classes.append({
                        'class': class_name,
                        'count': count,
                        'proportion': proportion
                    })

            if imbalanced_classes:
                imbalanced_features[feature_name] = imbalanced_classes

        print("\nImbalanced Classes (less than {:.1%} of samples):".format(threshold))
        if not imbalanced_features:
            print("  None detected.")

        for feature, classes in imbalanced_features.items():
            print(f"\n{feature}:")
            for cls in classes:
                print(f"  {cls['class']}: {cls['count']} samples ({cls['proportion']*100:.1f}%)")

        return imbalanced_features

    def create_validation_split(self, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15,
                                stratify_by=None, random_state=42):

        assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 0.001

        id_col = self.df.columns[0]
        image_ids = self.df[id_col].values

        stratify_labels = None
        if stratify_by is not None and stratify_by in self.df.columns:
            valid_mask = self.df[stratify_by].notna()
            if valid_mask.sum() < len(self.df):
                print(f" Stratification on '{stratify_by}' drops {len(self.df) - valid_mask.sum()} samples with missing labels.")
            ids_valid = self.df.loc[valid_mask, id_col].values
            labels_valid = self.df.loc[valid_mask, stratify_by].values

            train_ids, temp_ids, train_labels, temp_labels = train_test_split(
                ids_valid,
                labels_valid,
                train_size=train_ratio,
                random_state=random_state,
                stratify=labels_valid
            )

            val_ratio_adjusted = val_ratio / (val_ratio + test_ratio)
            val_ids, test_ids = train_test_split(
                temp_ids,
                train_size=val_ratio_adjusted,
                random_state=random_state,
                stratify=temp_labels
            )

            all_split_ids = set(train_ids) | set(val_ids) | set(test_ids)
            remaining_ids = set(image_ids) - all_split_ids
            if remaining_ids:
                print(f"[Info] {len(remaining_ids)} samples were not included in splits due to missing stratification labels. "
                      f"You may choose to assign them to train/val/test manually or use a different strat column.")

        else:
            # No stratification
            train_ids, temp_ids = train_test_split(
                image_ids,
                train_size=train_ratio,
                random_state=random_state
            )
            val_ratio_adjusted = val_ratio / (val_ratio + test_ratio)
            val_ids, test_ids = train_test_split(
                temp_ids,
                train_size=val_ratio_adjusted,
                random_state=random_state
            )

        splits = {
            'train': train_ids.tolist(),
            'val': val_ids.tolist(),
            'test': test_ids.tolist()
        }

        print("\nSplit Distribution:")
        print(f"Train: {len(train_ids)} samples ({len(train_ids)/len(image_ids)*100:.1f}%)")
        print(f"Validation: {len(val_ids)} samples ({len(val_ids)/len(image_ids)*100:.1f}%)")
        print(f"Test: {len(test_ids)} samples ({len(test_ids)/len(image_ids)*100:.1f}%)")

        if stratify_by is not None and stratify_by in self.df.columns:
            self.verify_stratification(splits, stratify_by)

        splits_path = os.path.join(self.output_dir, 'data_splits.json')
        with open(splits_path, 'w', encoding='utf-8') as f:
            json.dump(splits, f, ensure_ascii=False, indent=2)
        print(f"Splits saved to {splits_path}")

        return splits

    def verify_stratification(self, splits, feature_name):
        """Verify that class distribution is preserved in splits"""
        id_col = self.df.columns[0]
        df_indexed = self.df.set_index(id_col)

        print(f"\nClass distribution for '{feature_name}':")
        print("-" * 50)

        for split_name, split_ids in splits.items():
            split_df = df_indexed.loc[split_ids]
            value_counts = split_df[feature_name].value_counts(dropna=False, normalize=True)

            print(f"\n{split_name.capitalize()} split ({len(split_ids)} samples):")
            for value, proportion in value_counts.items():
                print(f"  {value}: {proportion*100:.1f}%")

    def visualize_distributions(self):
        """Create visualizations of feature distributions"""
        for feature_name, distribution in self.feature_distributions.items():
            counts = distribution['counts']
            labels = list(counts.keys())
            values = list(counts.values())

            plt.figure(figsize=(max(10, 0.25 * len(labels)), 6))
            bars = plt.bar(range(len(labels)), values)

            plt.xticks(range(len(labels)), labels, rotation=45, ha='right')
            plt.title(f'Distribution of {feature_name}', fontsize=12)
            plt.ylabel('Count')
            plt.tight_layout()

            for bar in bars:
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                         f'{int(height)}', ha='center', va='bottom')

            out_path = os.path.join(self.output_dir, f"{feature_name.replace(' ', '_')}_distribution.png")
            plt.savefig(out_path, dpi=150, bbox_inches='tight')
            plt.close()

        print(f"Visualizations saved to {self.output_dir}/")






In [5]:
analyzer = AnnotationAnalyzer(csv_path='Dataset//bender.csv', output_dir='visualizations')

df = analyzer.load_annotations()

distributions = analyzer.analyze_feature_distributions()

imbalanced = analyzer.identify_imbalanced_classes(threshold=0.10)

splits = analyzer.create_validation_split(
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    stratify_by='Sequence',   
    random_state=42
)

analyzer.visualize_distributions()



Dataset shape: (89, 16)
Columns: ['Case Number', 'الاسم', 'Gender', 'Grade', 'Age', 'Sequence', 'Drawing Position', 'Distance Between Drawings', 'Contact or Collision', 'Margin Usage', 'Paper Position Change', 'Gestalt Changes', 'ملاحظات دالة بشكل عام', 'التصنيفات المرضية', 'التعليق العام', 'Unnamed: 15']

First few rows:
   Case Number                       الاسم Gender                 Grade  Age  \
0            1         ياسين محمود علي حسن   Male  4th Grade Elementary    9   
1            2         عمرو احمد محمد فتحي   Male  4th Grade Elementary    9   
2            3  محمد سيد البدري عبد المطلب   Male  4th Grade Elementary    9   
3            4           عدي هاني سيد محمد   Male  4th Grade Elementary   10   
4            5        عمر حاتم عمر الفاروق   Male  4th Grade Elementary   11   

                Sequence Drawing Position Distance Between Drawings  \
0  Disorganized Sequence               No           0.75 to 1.25 cm   
1     Organized Sequence               No           0

Extract Bounding Boxes
model training:

In [ ]:
# from ultralytics import YOLO
#
# yaml_text = """
# path: Dataset
#
# train: train
# val: val
#
# nc: 9
# names:
#   0: figure_0
#   1: figure_1
#   2: figure_2
#   3: figure_3
#   4: figure_4
#   5: figure_5
#   6: figure_6
#   7: figure_7
#   8: figure_8
# """
#
# with open("dataset.yaml", "w") as f:
#     f.write(yaml_text)
#
# print("dataset.yaml written.")
#
# model = YOLO("yolov8n.pt")
#
# model.train(
#     data        = "dataset.yaml",
#     epochs      = 50,
#     imgsz       = 640,
#     batch       = 8,
#     patience    = 20, # early stopping
#     optimizer   = "AdamW",
#     lr0         = 1e-3,
#     augment     = True,
#     # augmentation for grayscale scans:
#     hsv_h       = 0.0, # no hue shift (grayscale)
#     hsv_s       = 0.0, # no saturation shift
#     hsv_v       = 0.3, # slight brightness variation
#     degrees     = 10,  # small rotation augmentation
#     translate   = 0.1,
#     scale       = 0.3,
#     project     = 'bender',
#     fliplr      = 0.0, # Bender figures are NOT flip-invariant
#     exist_ok    = True
# )

Extract Bender Figures function

In [4]:
from ultralytics import YOLO

FIGURE_NAMES = ["0","1","2","3","4","5","6","7","8"]

def extract_bender_figures(image_path,
                           model_path="runs/detect/bender/train/weights/best.pt",
                           conf_threshold=0.3,
                           padding=10):

    model = YOLO(model_path)
    img   = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")

    h, w = img.shape[:2]

    preds = model.predict(
        source    = image_path,
        conf      = conf_threshold,
        iou       = 0.4,          # NMS IoU threshold
        imgsz     = 640,
        verbose   = False
    )[0]                          # single image → first result

    boxes_out   = []
    classes_out = []
    names_out   = []
    scores_out  = []
    crops_out   = []

    if preds.boxes is not None:
        xyxy    = preds.boxes.xyxy.cpu().numpy().astype(int)
        cls_ids = preds.boxes.cls.cpu().numpy().astype(int)
        confs   = preds.boxes.conf.cpu().numpy()

        # sort detections left-to-right, top-to-bottom
        order = sorted(range(len(xyxy)),
                       key=lambda i: (xyxy[i][1] // 100, xyxy[i][0]))

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img

        for i in order:
            x1, y1, x2, y2 = xyxy[i]
            cls             = int(cls_ids[i])
            conf            = float(confs[i])

            # apply padding (clamp to image bounds)
            x1p = max(0,  x1 - padding)
            y1p = max(0,  y1 - padding)
            x2p = min(w,  x2 + padding)
            y2p = min(h,  y2 + padding)

            crop = gray[y1p:y2p, x1p:x2p]

            boxes_out.append((x1p, y1p, x2p, y2p))
            classes_out.append(cls)
            names_out.append(FIGURE_NAMES[cls] if cls < len(FIGURE_NAMES) else str(cls))
            scores_out.append(conf)
            crops_out.append(crop)

    print(f"Detected {len(boxes_out)} / 9 figures in {os.path.basename(image_path)}")

    return {
        "boxes"  : boxes_out,
        "classes": classes_out,
        "names"  : names_out,
        "scores" : scores_out,
        "crops"  : crops_out
    }

Draw boxes and visualize

In [36]:
import matplotlib.patches as patches

FIGURE_COLORS = {
    "0": "red",   "1": "orange", "2": "gold",   "3": "lime",
    "4": "cyan",  "5": "blue",   "6": "violet",  "7": "magenta", "8": "white"
}

def visualize_detections(image_path, results_dict):
    img     = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    ax.imshow(img_rgb)

    for box, name, score in zip(results_dict["boxes"],
                                results_dict["names"],
                                results_dict["scores"]):
        x1, y1, x2, y2 = box
        color = FIGURE_COLORS.get(name, "lime")
        rect  = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, f"Fig {name}  {score:.2f}",
                color=color, fontsize=9, fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.4, pad=1))

    ax.set_title(f"Bender Figures Detected: {len(results_dict['boxes'])} / 9  |  {os.path.basename(image_path)}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    # 3×3 grid of crops
    crops = results_dict["crops"]
    if crops:
        fig2, axes = plt.subplots(3, 3, figsize=(10, 10))
        for idx, ax2 in enumerate(axes.flat):
            if idx < len(crops):
                ax2.imshow(crops[idx], cmap="gray")
                ax2.set_title(f"Figure {results_dict['names'][idx]}  ({results_dict['scores'][idx]:.2f})")
            ax2.axis("off")
        plt.tight_layout()
        plt.show()

Testing Bounding Boxes

In [ ]:
# CELL 7: Run inference on 10 test images + summary report
import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from IPython.display import display, Image as IPImage

TEST_IMAGE_DIR = "Dataset/test"
MODEL_PATH     = "runs/detect/bender/train/weights/best.pt"
FIGURE_NAMES   = ["0", "1", "2", "3", "4", "5", "6", "7", "8"]
FIGURE_COLORS  = {
    "0": "red",    "1": "orange", "2": "gold",   "3": "lime",
    "4": "cyan",   "5": "blue",   "6": "violet",  "7": "magenta", "8": "white"
}

def show_inline(fig):
    """Render a matplotlib figure inline in PyCharm notebook output."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    buf.seek(0)
    display(IPImage(data=buf.read()))
    plt.close(fig)

test_images = sorted([
    os.path.join(TEST_IMAGE_DIR, f)
    for f in os.listdir(TEST_IMAGE_DIR)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

print(f"Found {len(test_images)} test images\n")

# ── run inference ─────────────────────────────────────────────────────────────
all_results  = {}
summary_rows = []

for img_path in test_images:
    results = extract_bender_figures(img_path, model_path=MODEL_PATH)
    all_results[img_path] = results

    detected_names = results["names"]
    missing        = [n for n in FIGURE_NAMES if n not in detected_names]

    summary_rows.append({
        "image"   : os.path.basename(img_path),
        "detected": len(detected_names),
        "missing" : ", ".join(missing) if missing else "—",
        "avg_conf": round(float(np.mean(results["scores"])), 3) if results["scores"] else 0.0
    })

# ── per-image visualizations ─────────────────────────────────────────────────
for img_path, results in all_results.items():
    img     = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # — annotated full image —
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    ax.imshow(img_rgb)
    for box, name, score in zip(results["boxes"], results["names"], results["scores"]):
        x1, y1, x2, y2 = box
        color = FIGURE_COLORS.get(name, "lime")
        rect  = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, f"Fig {name}  {score:.2f}",
                color=color, fontsize=9, fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.4, pad=1))
    ax.set_title(f"Detected: {len(results['boxes'])} / 9  |  {os.path.basename(img_path)}")
    ax.axis("off")
    plt.tight_layout()
    show_inline(fig)

    # — 3×3 crop grid —
    crops = results["crops"]
    if crops:
        fig2, axes = plt.subplots(3, 3, figsize=(10, 10))
        for idx, ax2 in enumerate(axes.flat):
            if idx < len(crops):
                ax2.imshow(crops[idx], cmap="gray")
                ax2.set_title(f"Figure {results['names'][idx]}  ({results['scores'][idx]:.2f})")
            ax2.axis("off")
        plt.tight_layout()
        show_inline(fig2)

# ── summary table ─────────────────────────────────────────────────────────────
summary_df = pd.DataFrame(summary_rows)
print("\n===== Inference Summary =====")
print(summary_df.to_string(index=False))

# ── bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
x_pos = range(len(summary_df))
ax.bar(x_pos, summary_df["detected"], color="steelblue")
ax.axhline(9, color="red", linestyle="--", label="Expected (9)")
ax.set_xlabel("Image")
ax.set_ylabel("Figures Detected")
ax.set_title("YOLOv8 Bender Figure Detection — Test Set")
ax.set_xticks(x_pos)
ax.set_xticklabels(summary_df["image"], rotation=45, ha="right")
ax.legend()
plt.tight_layout()
show_inline(fig)

Overlap Function

In [5]:
import cv2
import numpy as np
import pandas as pd
import os
from itertools import combinations


TOUCHING_THRESHOLD_PX = 3
CLOSE_THRESHOLD_PX = 15

TEMPLATE_DIR = "Dataset\\templates"


def load_templates():
    templates = {}

    for name in FIGURE_NAMES:
        path = os.path.join(TEMPLATE_DIR, f"{int(name) + 1}.png")

        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(path)

        # invert: black figure on white bg -> white figure on black bg
        binary = (img < 128).astype(np.uint8)

        cnts, _ = cv2.findContours(
            binary,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        if not cnts:
            raise ValueError(f"No contour found in template {name}")

        templates[name] = max(cnts, key=cv2.contourArea)

    return templates


TEMPLATES = load_templates()

def _binarize_full(gray):

    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    binary = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        31,
        8
    )

    binary = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        np.ones((2, 2), np.uint8)
    )

    return binary


# ------------------------------------------------------------------

def _build_ownership_masks(global_ink, boxes, names):

    H, W = global_ink.shape
    masks = []

    for (x1, y1, x2, y2), name in zip(boxes, names):

        crop = (global_ink[y1:y2, x1:x2] > 0).astype(np.uint8)

        n, labels, stats, _ = cv2.connectedComponentsWithStats(
            crop,
            connectivity=8
        )

        best_score = float("inf")
        best_mask = None

        template_cnt = TEMPLATES[name]

        for comp_id in range(1, n):

            comp = (labels == comp_id).astype(np.uint8)

            cnts, _ = cv2.findContours(
                comp,
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )

            if not cnts:
                continue

            cnt = max(cnts, key=cv2.contourArea)

            score = cv2.matchShapes(
                template_cnt,
                cnt,
                cv2.CONTOURS_MATCH_I1,
                0
            )

            if score < best_score:
                best_score = score
                best_mask = comp

        full = np.zeros((H, W), dtype=bool)

        if best_mask is not None:
            full[y1:y2, x1:x2] = best_mask.astype(bool)

        masks.append(full)

    return masks


# ------------------------------------------------------------------

def _min_ink_distance(mask_a, mask_b):
    if not mask_a.any() or not mask_b.any():
        return np.inf

    dt = cv2.distanceTransform(
        np.where(mask_a, 0, 255).astype(np.uint8),
        cv2.DIST_L2,
        5
    )

    return dt[mask_b].min()


# ------------------------------------------------------------------

def _classify_pair(mask_a, mask_b, box_a, box_b):

    # collision only if components physically intersect
    if np.any(mask_a & mask_b):
        return "Collision or Tendency"

    d = _min_ink_distance(mask_a, mask_b)

    if d <= 3:
        return "Drawing Overlap"

    return "No Overlap"


# ------------------------------------------------------------------

def check_figure_overlap(image, detection_result):
    if isinstance(image, str):
        gray = cv2.imread(image, 0)

    else:
        gray = image if image.ndim == 2 else cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    binary = _binarize_full(gray)
    cv2.imwrite("debug_binary.jpg", binary)

    boxes = detection_result["boxes"]

    masks = _build_ownership_masks(
        binary,
        detection_result["boxes"],
        detection_result["names"]
    )

    outputs = []

    for i, j in combinations(range(len(boxes)), 2):
        outputs.append(
            _classify_pair(
                masks[i],
                masks[j],
                boxes[i],
                boxes[j]
            )
        )

    return outputs


def overall_verdict(outputs):

    collision = outputs.count("Collision or Tendency")
    drawing = outputs.count("Drawing Overlap")

    if collision == 0 and drawing == 0:
        return "No Overlap"

    if collision >= drawing:
        return "Collision or Tendency"

    return "Drawing Overlap"


if __name__ == "__main__":

    # Load only the column named 'column_name' directly into a list
    my_list = pd.read_csv('Dataset/bender.csv', usecols=['Contact or Collision'])['Contact or Collision'].tolist()

    no = 0
    yes = 0
    ten = 0
    acc = 0
    for i in range(1,90):

        img_path = f"Dataset/Preprocessed/{i}.jpg"
        det = extract_bender_figures(img_path)

        pair_results = check_figure_overlap(img_path, det)
        verdict = overall_verdict(pair_results)

        print(f"\n{'='*45}")
        print(f"  Image   : {os.path.basename(img_path)}")
        print(f"  VERDICT : {verdict}")
        print(f"  ACTUAL  : {my_list[i - 1]}")
        if verdict == my_list[i - 1]:
            acc += 1
        if verdict == "Drawing Overlap":
            yes += 1
        elif verdict == "No Overlap":
            no += 1
        elif verdict == "Collision or Tendency":
            ten += 1
        print(f"{'='*45}")

    print(f"Collision or Tendency: {ten / 89 * 100}%")
    print(f"Drawing Overlap: {yes / 89 * 100}%")
    print(f"No Overlap: {no / 89 * 100}%")
    print(f"ACCURACY: {acc / 89 * 100}%")

Detected 9 / 9 figures in 1.jpg

  Image   : 1.jpg
  VERDICT : No Overlap
  ACTUAL  : No Overlap
Detected 9 / 9 figures in 2.jpg

  Image   : 2.jpg
  VERDICT : No Overlap
  ACTUAL  : No Overlap
Detected 9 / 9 figures in 3.jpg

  Image   : 3.jpg
  VERDICT : No Overlap
  ACTUAL  : No Overlap
Detected 9 / 9 figures in 4.jpg

  Image   : 4.jpg
  VERDICT : No Overlap
  ACTUAL  : No Overlap
Detected 9 / 9 figures in 5.jpg

  Image   : 5.jpg
  VERDICT : No Overlap
  ACTUAL  : Collision or Tendency
Detected 9 / 9 figures in 6.jpg

  Image   : 6.jpg
  VERDICT : No Overlap
  ACTUAL  : No Overlap
Detected 8 / 9 figures in 7.jpg

  Image   : 7.jpg
  VERDICT : Drawing Overlap
  ACTUAL  : Collision or Tendency
Detected 9 / 9 figures in 8.jpg

  Image   : 8.jpg
  VERDICT : Collision or Tendency
  ACTUAL  : No Overlap
Detected 9 / 9 figures in 9.jpg

  Image   : 9.jpg
  VERDICT : Collision or Tendency
  ACTUAL  : No Overlap
Detected 9 / 9 figures in 10.jpg

  Image   : 10.jpg
  VERDICT : No Overlap
  

Margin Usage

In [5]:
# MARGIN USAGE

import cv2
import os

def analyze_margin_usage(results, image_path, pixels_per_cm=37.8):
    """
    Margin Usage Rule:
    TRUE if 4 or more figures are within
    0.75 cm of any page margin.
    Returns:
        dict containing margin usage analysis
    """

    # Load image
    img = cv2.imread(image_path)

    if img is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")

    h, w = img.shape[:2]


    # Figure is considered near margin if within 0.75 cm
    margin_threshold_cm = 0.75

    # Convert cm → pixels
    threshold_px = int(margin_threshold_cm * pixels_per_cm)

    # Extract detected figure boxes
    boxes = results.get("boxes", [])

    near_margin_count = 0
    margin_figures = []


    # Check each figure against page margins
    for i, (x1, y1, x2, y2) in enumerate(boxes):

        near_margin = False
        touched = []

        # Left margin
        if x1 <= threshold_px:
            near_margin = True
            touched.append("left")

        # Top margin
        if y1 <= threshold_px:
            near_margin = True
            touched.append("top")

        # Right margin
        if x2 >= (w - threshold_px):
            near_margin = True
            touched.append("right")

        # Bottom margin
        if y2 >= (h - threshold_px):
            near_margin = True
            touched.append("bottom")

        # Store figures near margins
        if near_margin:

            near_margin_count += 1

            margin_figures.append({
                "figure": results["names"][i]
                if i < len(results.get("names", []))
                else "unknown",

                "box": (x1, y1, x2, y2),

                "near_margins": touched
            })


    # FINAL DECISION RULE
    margin_usage = near_margin_count >= 4

    # Return Results
    return {

        "margin_usage": margin_usage,
        "margin_usage_label":
            "Yes" if margin_usage else "No",
        "near_margin_count": near_margin_count,
        "threshold_px": threshold_px,
        "margin_threshold_cm": margin_threshold_cm,
        "image_width": w,
        "image_height": h,
        "figures_near_margin": margin_figures
    }



# PROCESS DATASET
folder_path = "Dataset/Preprocessed"

all_results = []

for img_name in sorted(os.listdir(folder_path)):

    if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    image_path = os.path.join(folder_path, img_name)

    # Step 1: Extract Bender figures
    results = extract_bender_figures(
        image_path=image_path
    )

    # Step 2: Analyze margin usage
    margin_results = analyze_margin_usage(
        results,
        image_path=image_path,
        pixels_per_cm=37.8
    )

    # Store combined output
    all_results.append({
        "image": img_name,
        **margin_results
    })

    print(f"Processed: {img_name}")



# DISPLAY RESULTS
import pandas as pd
from IPython.display import display

df = pd.DataFrame(all_results)

display(df)

Detected 9 / 9 figures in 1.jpg
Processed: 1.jpg
Detected 9 / 9 figures in 10.jpg
Processed: 10.jpg
Detected 9 / 9 figures in 11.jpg
Processed: 11.jpg
Detected 10 / 9 figures in 12.jpg
Processed: 12.jpg
Detected 9 / 9 figures in 13.jpg
Processed: 13.jpg
Detected 9 / 9 figures in 14.jpg
Processed: 14.jpg
Detected 9 / 9 figures in 15.jpg
Processed: 15.jpg
Detected 9 / 9 figures in 16.jpg
Processed: 16.jpg
Detected 8 / 9 figures in 17.jpg
Processed: 17.jpg
Detected 10 / 9 figures in 18.jpg
Processed: 18.jpg
Detected 9 / 9 figures in 19.jpg
Processed: 19.jpg
Detected 9 / 9 figures in 2.jpg
Processed: 2.jpg
Detected 9 / 9 figures in 20.jpg
Processed: 20.jpg
Detected 9 / 9 figures in 21.jpg
Processed: 21.jpg
Detected 9 / 9 figures in 22.jpg
Processed: 22.jpg
Detected 9 / 9 figures in 23.jpg
Processed: 23.jpg
Detected 9 / 9 figures in 24.jpg
Processed: 24.jpg
Detected 9 / 9 figures in 25.jpg
Processed: 25.jpg
Detected 9 / 9 figures in 26.jpg
Processed: 26.jpg
Detected 9 / 9 figures in 27.jpg


,image,margin_usage,margin_usage_label,near_margin_count,threshold_px,margin_threshold_cm,image_width,image_height,figures_near_margin
0,1.jpg,False,No,2,28,0.75,1241,1755,"[{'figure': '5', 'box': (754, 1369, 1059, 1755..."
1,10.jpg,False,No,1,28,0.75,1241,1755,"[{'figure': '8', 'box': (111, 1547, 809, 1755)..."
2,11.jpg,False,No,3,28,0.75,1241,1755,"[{'figure': '0', 'box': (439, 20, 746, 505), '..."
3,12.jpg,False,No,0,28,0.75,1241,1755,[]
4,13.jpg,False,No,0,28,0.75,1241,1755,[]
...,...,...,...,...,...,...,...,...,...
84,86.jpg,False,No,0,28,0.75,1241,1755,[]
85,87.jpg,False,No,1,28,0.75,1755,1241,"[{'figure': '6', 'box': (0, 753, 693, 1190), '..."
86,88.jpg,False,No,1,28,0.75,1241,1755,"[{'figure': '6', 'box': (9, 0, 720, 490), 'nea..."
87,89.jpg,False,No,1,28,0.75,1241,1755,"[{'figure': '6', 'box': (0, 870, 590, 1293), '..."


Change Paper Position

In [6]:
import cv2
import numpy as np


def analyze_paper_position_change(
    image_path,
    angle_threshold=15
):
    """
    Paper Position Change Rule:
    TRUE if estimated page/sketch rotation
    exceeds the threshold angle.

    Returns:
        dict containing:
            - paper position change decision
            - estimated rotation angle
            - detected line statistics
    """

    # LOAD IMAGE
    img = cv2.imread(image_path)

    if img is None:
        raise FileNotFoundError(
            f"Could not load image: {image_path}"
        )

    original = img.copy()


    # PREPROCESSING
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Edge detection
    edges = cv2.Canny(
        blurred,
        threshold1=50,
        threshold2=150
    )


    # DETECT LINES
    lines = cv2.HoughLinesP(
        edges,
        rho=1,
        theta=np.pi / 180,
        threshold=80,
        minLineLength=50,
        maxLineGap=10
    )

    if lines is None:

        return {
            "paper_position_change": False,
            "paper_position_change_label": "No",
            "rotation_angle": 0.0,
            "detected_lines": 0,
            "reason": "No significant lines detected"
        }


    # COMPUTE LINE ANGLES
    angles = []

    for line in lines:

        x1, y1, x2, y2 = line[0]

        angle = np.degrees(
            np.arctan2(
                (y2 - y1),
                (x2 - x1)
            )
        )

        # Normalize angle to [-90, 90]
        if angle > 90:
            angle -= 180

        if angle < -90:
            angle += 180

        angles.append(angle)


    # ESTIMATE GLOBAL ROTATION
    # Median is more robust against outliers
    rotation_angle = float(np.median(angles))


    # FINAL DECISION RULE
    paper_position_change = (
        abs(rotation_angle) > angle_threshold
    )

    # RETURN RESULTS
    return {

        "paper_position_change":
            paper_position_change,

        "paper_position_change_label":
            "Yes"
            if paper_position_change
            else "No",

        "rotation_angle":
            round(rotation_angle, 2),

        "angle_threshold":
            angle_threshold,

        "detected_lines":
            len(lines),

        "image_width":
            img.shape[1],

        "image_height":
            img.shape[0]
    }


# PROCESS DATASET
folder_path = "Dataset/Preprocessed"

all_position_results = []

for img_name in sorted(os.listdir(folder_path)):

    if not img_name.lower().endswith(
        (".jpg", ".png", ".jpeg")
    ):
        continue

    image_path = os.path.join(
        folder_path,
        img_name
    )

    # Analyze paper position change
    position_results = analyze_paper_position_change(
        image_path=image_path,
        angle_threshold=15
    )

    all_position_results.append({
        "image": img_name,
        **position_results
    })

    print(f"Processed: {img_name}")


# DISPLAY RESULTS
import pandas as pd
from IPython.display import display

df_position = pd.DataFrame(all_position_results)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
display(df_position)

Processed: 1.jpg
Processed: 10.jpg
Processed: 11.jpg
Processed: 12.jpg
Processed: 13.jpg
Processed: 14.jpg
Processed: 15.jpg
Processed: 16.jpg
Processed: 17.jpg
Processed: 18.jpg
Processed: 19.jpg
Processed: 2.jpg
Processed: 20.jpg
Processed: 21.jpg
Processed: 22.jpg
Processed: 23.jpg
Processed: 24.jpg
Processed: 25.jpg
Processed: 26.jpg
Processed: 27.jpg
Processed: 28.jpg
Processed: 29.jpg
Processed: 3.jpg
Processed: 30.jpg
Processed: 31.jpg
Processed: 32.jpg
Processed: 33.jpg
Processed: 34.jpg
Processed: 35.jpg
Processed: 36.jpg
Processed: 37.jpg
Processed: 38.jpg
Processed: 39.jpg
Processed: 4.jpg
Processed: 40.jpg
Processed: 41.jpg
Processed: 42.jpg
Processed: 43.jpg
Processed: 44.jpg
Processed: 45.jpg
Processed: 46.jpg
Processed: 47.jpg
Processed: 48.jpg
Processed: 49.jpg
Processed: 5.jpg
Processed: 50.jpg
Processed: 51.jpg
Processed: 52.jpg
Processed: 53.jpg
Processed: 54.jpg
Processed: 55.jpg
Processed: 56.jpg
Processed: 57.jpg
Processed: 58.jpg
Processed: 59.jpg
Processed: 6.jp

,image,paper_position_change,paper_position_change_label,rotation_angle,angle_threshold,detected_lines,image_width,image_height
0,1.jpg,False,No,-4.57,15,27,1241,1755
1,10.jpg,False,No,-8.18,15,44,1241,1755
2,11.jpg,False,No,-4.90,15,28,1241,1755
3,12.jpg,True,Yes,29.69,15,28,1241,1755
4,13.jpg,False,No,-2.09,15,100,1241,1755
5,14.jpg,False,No,-2.66,15,49,1241,1755
6,15.jpg,False,No,-12.80,15,19,1241,1755
7,16.jpg,False,No,-1.06,15,103,1241,1755
8,17.jpg,True,Yes,-56.96,15,52,1241,1755
9,18.jpg,False,No,0.00,15,81,1241,1755
